# What will the economic growth be in the G20 countries during the second quarter (Q2) 2025, compared with the quarter before, based on real GDP?

In [ ]:
%load_ext autoreload
%autoreload 2

In [8]:
import os
from list_active_ifps import list_active_ifps
from save_ifps_to_disk import save_ifps_to_disk
from forecast_ifp import forecast_ifp
from gather_news_for_ifps import gather_news_for_ifps
from detailed_proposition import detailed_proposition
from wiki_semantic_search import wiki_semantic_search
from split_news_into_text_and_urls import split_news_into_text_and_urls
from create_source_summaries import create_source_summaries
from rephrase_binary_outcomes import rephrase_binary_outcomes
from format_research import format_research
from glimt_forecast_prompt import glimt_forecast_prompt
from humor_me import humor_me
from get_forecast_components import *
from median_forecast import median_forecast
from median_rationale import median_rationale
from jsx_request import jsx_request
from jsx_forecast import jsx_forecast
from datetime import datetime

In [2]:
ifps = list_active_ifps()

200


In [3]:
id_to_ifp = save_ifps_to_disk(ifps)

In [4]:
ifp = [ifp for ifp in ifps if ifp['id'] == 501][0]

In [7]:
news = gather_news_for_ifps([ifp])

saved glimt/news/501.txt


In [9]:
print('begin FORECASTING', ifp['id'], ifp['props']['title'], datetime.now())
title_plus_criteria = detailed_proposition(ifp)
wiki_articles = wiki_semantic_search(title_plus_criteria)

begin FORECASTING 501 What will the economic growth be in the G20 countries during the second quarter (Q2) 2025, compared with the quarter before, based on real GDP? 2025-07-28 15:56:13.270178


In [14]:
print('\n'.join(sorted([y for x,y,z in wiki_articles])))

GDP Deflator
GDP gap
List of countries by GDP
List of countries by GDP (nominal) (2004-2005)
List of countries by GDP (ppp) per capita growth rate
List of countries by GDP (real) growth rate
List of countries by GDP (real) growth rate per capita
List of countries by real GDP
List of countries by real GDP growth rate
List of countries by real GDP growth rate (2009)


In [16]:
ifp_news_sources, ifp_news_text = split_news_into_text_and_urls(ifp, news)

In [ ]:
source_summaries, sources = create_source_summaries(ifp['id'], title_plus_criteria, wiki_articles, ifp_news_sources, ifp_news_text)

In [19]:
fn2 = f'glimt/source_summaries/{ifp['id']}_combined.txt' # source summaries
fn2

'glimt/source_summaries/501_combined.txt'

In [20]:
    with open(fn2, 'w') as f:
        f.write('\n'.join(source_summaries))

In [ ]:
rephrase_binary_outcomes(ifp)
research = format_research(source_summaries)

In [42]:
prompt, rejected = glimt_forecast_prompt(ifp, research)

In [47]:
prompt_tries = 1 # Mistral 4 bit has no randomness
answers = [humor_me(prompt, i+1) for i in range(prompt_tries)]
binProbs = [get_bin_probs(a) for a in answers]
rights = [get_rights(a) for a in answers]
wrongs = [get_wrongs(a) for a in answers]

START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.15341682036717733

```binProbs
[0.1, 0.3, 0.4, 0.2]
```

```rRight
The probabilities are based on the historical trend of consistent economic growth in G20 countries during 2024, with quarterly increases ranging from 0.7% to 0.9%. The OECD report expected on September 15, 2025, is likely to provide up-to-date data that reflects similar growth patterns. The inclusion of a reasonable range of probabilities (0.5% to 1% being the most likely) accounts for potential variations due to economic policies, global events, and other influencing factors.
```

```rWrong
The probabilities might be incorrect due to the lack of specific quarterly data for Q2 2025 in the provided content. The historical trends from 2024 may not accurately predict future growth, especially if there are significant changes in economic policies, geopolitical events, or other external factors. Additionally, the r

In [48]:
## Median forecasts and rationales
forecast = rejected + (median_forecast(binProbs) if len(binProbs) > 1 else binProbs[0])

right = median_rationale(rights) if len(rights) > 1 else rights[0]
wrong = median_rationale(wrongs) if len(wrongs) > 1 else wrongs[0]
jsx_request(jsx_forecast(ifp['id'],forecast,right,wrong,sources))
print('end FORECASTING', ifp['id'], ifp['props']['title'], datetime.now())

result = (forecast, right, wrong, sources)
fn = f'glimt/forecast'
import os
os.makedirs(fn, exist_ok=True)
fn = f"{fn}/{ifp['id']}.json"

import json
with open(fn, 'w') as f:
    json.dump(result, f)

200
end FORECASTING 501 What will the economic growth be in the G20 countries during the second quarter (Q2) 2025, compared with the quarter before, based on real GDP? 2025-07-28 20:13:46.356518


In [49]:
crowd = [0.02, 0.32, 0.59, 0.07]

In [51]:
forecast

[0.1, 0.3, 0.4, 0.2]

In [52]:
import numpy as np

In [53]:
np.linalg.norm(np.array(crowd)-np.array(forecast))

0.24454038521274962

In [55]:
perp1 = [0.07, 0.19, 0.53, 0.21]

In [57]:
perp2 = [0.0, 0.0, 0.55, 0.45]

In [59]:
grok1 = [0.10, 0.25, 0.45, 0.20]

In [61]:
grok2 = [0.15, 0.30, 0.40, 0.15]

In [62]:
np.linalg.norm(np.array(crowd)-np.array(grok2))

0.24454038521274962